In [1]:
import sys
print(sys.executable)

c:\Users\Lello\Documents\GitHub\ai-ghostwriter-dante\.venv\Scripts\python.exe


In [2]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

print("TensorFlow:", tf.__version__)

TensorFlow: 2.21.0


# Prototipo originale Shakespeare

In questa fase eseguo il codice originale fornito dal corso senza modifiche, per analizzare il dataset e il numero di caratteri unici utilizzati dal modello.

In [3]:
import os
os.environ["KERAS_BACKEND"] = "torch"

import keras
import tensorflow as tf
import numpy as np

keras.mixed_precision.set_global_policy("mixed_float16")

path_to_file = tf.keras.utils.get_file(
    'shakespeare.txt',
    'https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt'
)

text = open(path_to_file, 'rb').read().decode(encoding='utf-8')

vocab = sorted(set(text))

print("Numero caratteri unici Shakespeare:", len(vocab))
print("Primi 20 caratteri:", vocab[:20])

1115394/1115394 ━━━━━━━━━━━━━━━━━━━━ 1s 1us/step
Numero caratteri unici Shakespeare: 65
Primi 20 caratteri: ['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G']


## Analisi del dataset Shakespeare

Il dataset Tiny Shakespeare contiene 65 caratteri unici.

Questo valore rappresenta la dimensione del vocabolario utilizzato dal modello per la generazione del testo.

# Cambio dataset: Divina Commedia

In questa fase sostituisco il dataset Tiny Shakespeare con il testo della Divina Commedia di Dante Alighieri, usando la libreria requests come richiesto dalla traccia.

In [4]:
import requests
import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

def scarica_testo_dante():
    """
    Scarica il testo della Divina Commedia usando requests.
    Imposta encoding UTF-8 per gestire correttamente gli accenti italiani.
    """
    url = "https://dmf.unicatt.it/~della/pythoncourse18/commedia.txt"
    print(f"Scaricamento testo da: {url}...")

    try:
        response = requests.get(url, verify=False)
        response.raise_for_status()
        response.encoding = "utf-8"

        text = response.text

        print(f"Download completato. Lunghezza testo: {len(text)} caratteri")
        print("Esempio primi 300 caratteri:")
        print(text[:300])

        return text

    except Exception as e:
        print(f"Errore nel download principale: {e}")
        return None

text_dante = scarica_testo_dante()

Scaricamento testo da: https://dmf.unicatt.it/~della/pythoncourse18/commedia.txt...
Download completato. Lunghezza testo: 551846 caratteri
Esempio primi 300 caratteri:
LA DIVINA COMMEDIA
di Dante Alighieri
INFERNO



Inferno: Canto I

  Nel mezzo del cammin di nostra vita
mi ritrovai per una selva oscura
ché la diritta via era smarrita.
  Ahi quanto a dir qual era è cosa dura
esta selva selvaggia e aspra e forte
che nel pensier rinova la paura!
  Tan


In [6]:
vocab_dante = sorted(set(text_dante))

print("Numero caratteri unici Dante:", len(vocab_dante))
print("Primi 20 caratteri:", vocab_dante[:20])

Numero caratteri unici Dante: 69
Primi 20 caratteri: ['\n', '\r', ' ', '!', '"', "'", '(', ')', ',', '-', '.', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F']


## Confronto vocabolario

- Shakespeare: 65 caratteri unici
- Dante: 69 caratteri unici

Il testo della Divina Commedia presenta un vocabolario leggermente più ampio rispetto al dataset Tiny Shakespeare, probabilmente per la presenza di caratteri tipici della lingua italiana e di una diversa struttura testuale.

In [7]:
# Vocabolario Dante
vocab = sorted(set(text_dante))

# Dizionari di conversione
char2idx = {u: i for i, u in enumerate(vocab)}
idx2char = np.array(vocab)

# Conversione testo -> numeri
text_as_int = np.array([char2idx[c] for c in text_dante])

print("Dimensione vocabolario:", len(vocab))
print("Lunghezza testo numerico:", len(text_as_int))
print("Primi 20 valori:", text_as_int[:20])

Dimensione vocabolario: 69
Lunghezza testo numerico: 551846
Primi 20 valori: [23 14  2 17 22 33 22 25 14  2 16 26 24 24 18 17 22 14  1  0]


In [8]:
# Lunghezza sequenza ridotta per Dante
seq_length = 80

char_dataset = tf.data.Dataset.from_tensor_slices(text_as_int)

sequences = char_dataset.batch(seq_length + 1, drop_remainder=True)

def split_input_target(chunk):
    input_text = chunk[:-1]
    target_text = chunk[1:]
    return input_text, target_text

dataset = sequences.map(split_input_target)

BATCH_SIZE = 128
BUFFER_SIZE = 10000

dataset = (
    dataset
    .cache()
    .shuffle(BUFFER_SIZE)
    .batch(BATCH_SIZE, drop_remainder=True)
    .prefetch(tf.data.AUTOTUNE)
)

print("Dataset Dante creato")
print("seq_length =", seq_length)

Dataset Dante creato
seq_length = 80


In [9]:
vocab_size = len(vocab)
embedding_dim = 256
rnn_units = 512

def build_model(vocab_size, embedding_dim, rnn_units):
    model = keras.Sequential([
        keras.layers.Embedding(vocab_size, embedding_dim),
        keras.layers.GRU(
            rnn_units,
            return_sequences=True,
            recurrent_initializer="glorot_uniform"
        ),
        keras.layers.Dense(vocab_size)
    ])
    return model

model = build_model(vocab_size, embedding_dim, rnn_units)

model.compile(
    optimizer="adam",
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True)
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [10]:
history = model.fit(
    dataset,
    epochs=3
)

Epoch 1/3
53/53 ━━━━━━━━━━━━━━━━━━━━ 128s 2s/step - loss: 2.8005
Epoch 2/3
53/53 ━━━━━━━━━━━━━━━━━━━━ 132s 2s/step - loss: 2.1273
Epoch 3/3
53/53 ━━━━━━━━━━━━━━━━━━━━ 128s 2s/step - loss: 1.9505


In [11]:
history_2 = model.fit(
    dataset,
    epochs=7
)

Epoch 1/7
53/53 ━━━━━━━━━━━━━━━━━━━━ 112s 2s/step - loss: 1.8393
Epoch 2/7
53/53 ━━━━━━━━━━━━━━━━━━━━ 117s 2s/step - loss: 1.7564
Epoch 3/7
53/53 ━━━━━━━━━━━━━━━━━━━━ 116s 2s/step - loss: 1.6882
Epoch 4/7
53/53 ━━━━━━━━━━━━━━━━━━━━ 116s 2s/step - loss: 1.6292
Epoch 5/7
53/53 ━━━━━━━━━━━━━━━━━━━━ 128s 2s/step - loss: 1.5809
Epoch 6/7
53/53 ━━━━━━━━━━━━━━━━━━━━ 125s 2s/step - loss: 1.5406
Epoch 7/7
53/53 ━━━━━━━━━━━━━━━━━━━━ 119s 2s/step - loss: 1.5051


# Generazione testo e tuning della temperatura

Dopo l'addestramento del modello sul testo della Divina Commedia, vengono effettuati tre test di generazione con temperature differenti.

- Temperatura 0.2: comportamento conservativo
- Temperatura 1.0: comportamento standard
- Temperatura 2.0: comportamento creativo

In [13]:
def generate_text(model, start_string, temperature=1.0, num_generate=500):
    input_eval = [char2idx[s] for s in start_string]
    input_eval = tf.expand_dims(input_eval, 0)

    text_generated = []

    for _ in range(num_generate):
        predictions = model(input_eval)
        predictions = predictions[:, -1, :] / temperature

        predicted_id = tf.random.categorical(predictions, num_samples=1)[0, 0].numpy()

        input_eval = tf.concat(
            [input_eval, tf.expand_dims([predicted_id], 0)],
            axis=1
        )

        if input_eval.shape[1] > seq_length:
            input_eval = input_eval[:, 1:]

        text_generated.append(idx2char[predicted_id])

    return start_string + "".join(text_generated)

In [14]:
testo_temp_02 = generate_text(
    model,
    start_string="Nel mezzo ",
    temperature=0.2,
    num_generate=500
)

print(testo_temp_02)

Nel mezzo de la spera scolta
de la mente che si facea in su la sua grande
che la mani a la bella schiera fatto.
  E io a la sua parte si stanno
che si facea in su la parte a la sua vista,
  così la calar di quella contenta
che la parte per la vista di color ch'altra volta.
  Ma perché l'altro più di suo parlar s'affetto
che si fece di men di color di son si spesa,
  che l'un di quella parte che 'l suo passo,
che li occhi suoi con la via che si scessa
la mania di fuor de la sua parte si parte.



In [15]:
testo_temp_10 = generate_text(
    model,
    start_string="Nel mezzo ",
    temperature=1.0,
    num_generate=500
)

print(testo_temp_10)

Nel mezzo ne la dita,
che a l'un'amitor lo prodo scola
rispuose il tuo loco e 'i queli ti presso
rietrapeso lo 'mperto e tuola,
là giùsitto poi, perché obbe brona
che 'l cagion sì come per le 'mpeti,
di' sì come ciascun dirmituto è da,
vai lamingar di cero d'uno a piega
così limera fé me per tempo,
morte ch'i' vissi fu sgradi, e quivi torti
di quai di queto scielo, ch'avvien che 'n s'erama
lo lucere a che saletta i mente serma,
quali mi presa tarda da l'altore avere,
se non riguardo il prima 


In [16]:
testo_temp_20 = generate_text(
    model,
    start_string="Nel mezzo ",
    temperature=2.0,
    num_generate=500
)

print(testo_temp_20)

Nel mezzo cBeun, poite"'chi ellazi to.
sì ca come mivo, evanzi E orno
potOqaò, adque?bo,
  unde tidm'inatanno", làfgra";
frotdimemonzieniFNoi hai quini: fam,   zì l'ul fosso
ell'aspu, S: oo!!R; Che l: Uir dì 'amvarbi piacque
bené comma ne'bentolamvollo,
non ladAzembitròò lebbbatà,
perà non aurlleE, odbile, in zas"e.
  e .
  Paràllò glóraia vira
Gel iungrezzo ilrurre Dlpo tante a'Lada"O.
  QAuanebà vépafcurameszo.
 he fede mezzo Fòudriesutbe itpUTan,
quantovro,: suo vappi!; e 'nxotedovo alpru


# Conclusioni

## Confronto tra i dataset

- Shakespeare: 65 caratteri unici
- Dante: 69 caratteri unici

Il testo della Divina Commedia presenta un vocabolario leggermente più ampio rispetto al dataset Tiny Shakespeare.

## Modifiche effettuate

- Sostituito il dataset Shakespeare con la Divina Commedia.
- Utilizzato requests per il download del testo.
- Impostato encoding UTF-8 per la corretta gestione degli accenti.
- Ridotta la lunghezza della sequenza da 250 a 80 caratteri.
- Addestrato il modello per 10 epoche.

## Analisi della temperatura

### Temperatura 0.2
Testo grammaticalmente più stabile ma molto ripetitivo.

### Temperatura 1.0
Miglior compromesso tra coerenza e creatività. Il testo mantiene una struttura simile allo stile dantesco pur introducendo variazioni.

### Temperatura 2.0
Testo più creativo ma spesso incoerente e con parole inventate.

## Temperatura migliore

La temperatura 1.0 ha prodotto il risultato più credibile e bilanciato.

## Miglior testo generato

Nel mezzo ne la dita,
che a l'un'amitor lo prodo scola
rispuose il tuo loco e 'i queli ti presso
rietrapeso lo 'mperto e tuola,
là giùsitto poi, perché obbe brona
che 'l cagion sì come per le 'mpeti,
di' sì come ciascun dirmituto è da,
vai lamingar di cero d'uno a piega
così limera fé me per tempo,
morte ch'i' vissi fu sgradi, e quivi torti
di quai di queto scielo, ch'avvien che 'n s'erama
lo lucere a che saletta i mente serma,
quali mi presa tarda da l'altore avere,
se non riguardo il prima